# 07 — Gerar OOF/holdout | PharmaDemand — sem bloqueio por divergência D+7/D+15

Esta versão **não soma** os dois horizontes e não altera os valores originais de `quantity` em `df`.

**Atenção:** o modo provisório `prefer_horizon=7` apenas contorna o bloqueio do histórico para testes. Não comprova a semântica do alvo; para métricas confiáveis do Ridge é necessário usar a série real `salesdaily` e confirmar se `quantity` é venda diária em `date+h`, venda na origem, ou acumulado do horizonte.


In [17]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / 'src').is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Abra este notebook dentro da pasta do PharmaDemand com src/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models import generate_stacking_predictions as stacking

# O ROOT do modulo depende da sua posicao em src/models/.
print('ROOT:', stacking.ROOT)
print('INPUT:', stacking.FEATURES_FILE)
print('OUTPUT:', stacking.OUTPUT_DIR)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ROOT: D:\PROJETOS\git_repo\PharmaDemand-ml
INPUT: D:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet
OUTPUT: D:\PROJETOS\git_repo\PharmaDemand-ml\data\stacking


## 1. Carregar dados supervisionados originais

Não reutilize o `df` que sofreu `update()` ou soma de D+7/D+15 no notebook antigo. Esta etapa **não corrige** `quantity` do dataset supervisionado. Ela só prepara a série diária auxiliar para SARIMAX e AutoTheta.


In [18]:
import pandas as pd

# Recarregar: evita carregar quantidades contaminadas pelo df.update()
df = pd.read_parquet(stacking.FEATURES_FILE).copy()
df['date'] = pd.to_datetime(df['date']).dt.normalize()
df = df.loc[
    df['split'].isin(['train', 'test'])
    & df['horizon_days'].isin([7, 15])
].copy()

KEYS = ['date', 'category', 'horizon_days']
if df.duplicated(KEYS).any():
    raise ValueError('Existem duplicatas por date/category/horizon_days no dataset supervisionado.')

print('Quantidade de linhas:', len(df))
print(df['split'].value_counts(dropna=False))
print(df['horizon_days'].value_counts(dropna=False))
display(df.loc[(df['date'] == '2014-03-18') & (df['category'] == 'N02BE'),
               KEYS + ['split','quantity']])


Quantidade de linhas: 4078
split
train    3712
test      366
Name: count, dtype: int64
horizon_days
7     2043
15    2035
Name: count, dtype: int64


,date,category,horizon_days,split,quantity
12,2014-03-18,N02BE,7,train,20.1
2047,2014-03-18,N02BE,15,train,20.1


In [19]:
# Coloque os arquivos CORRIGIDOS na pasta src/models/ com os nomes originais.
import importlib
from src.models import prepare_stacking_history as prep
importlib.reload(prep)
importlib.reload(stacking)

FILL_METHOD = 'ffill'

# Solução exploratória: escolhe D+7 em datas-alvo coincidentes (sem média/soma).
# PREFERÍVEL: passar history_df com vendas diárias REAIS (veja próximo comentário).
daily, gaps_audit = prep.build_daily_history(
    df, fill_method=FILL_METHOD, prefer_horizon=7
)

# Se houver df_salesdaily ORIGINAL carregado (colunas datum e categorias ATC),
# substitua a chamada acima por:
# raw = df_salesdaily.rename(columns={'datum': 'date'})
# sales_long = raw.melt(id_vars='date', value_vars=df['category'].unique(),
#                       var_name='category', value_name='quantity')
# daily, gaps_audit = prep.build_daily_history(
#     df, fill_method=FILL_METHOD, history_df=sales_long
# )

conflicts = prep.target_conflicts(df)
print('Conflitos D+7/D+15 na mesma data-alvo:', len(conflicts))
display(conflicts.head(10))
print('Histórico diário:', daily.shape)
display(gaps_audit)

HISTORY_FILE = ROOT / 'data' / 'features' / 'stacking_daily_history.parquet'
HISTORY_FILE.parent.mkdir(parents=True, exist_ok=True)
daily.to_parquet(HISTORY_FILE, index=False)
stacking.DAILY_HISTORY_FILE = HISTORY_FILE

# Libera a execução SEM VALIDAR o alinhamento do alvo.
# NÃO reportar métricas resultantes como válidas até revisar os targets!
stacking.ENFORCE_TARGET_ALIGNMENT = False
print('Modo exploratório; ENFORCE_TARGET_ALIGNMENT =', stacking.ENFORCE_TARGET_ALIGNMENT)


Conflitos D+7/D+15 na mesma data-alvo: 2013


C:\Users\AlexK\AppData\Local\Temp\ipykernel_30356\3622088872.py:11: RuntimeWarning: 2013 datas-alvo possuem alvos D+7/D+15 divergentes. A divergencia foi REGISTRADA, nao corrigida; verifique a semantica dos alvos antes de interpretar metricas do ensemble.
category target_date  min_quantity  max_quantity  num_horizons  difference
   N02BE  2017-01-14        34.625         161.0             2     126.375
   N02BE  2017-01-06        38.400         161.0             2     122.600
   N02BE  2016-01-16         0.000          88.3             2      88.300
  daily, gaps_audit = prep.build_daily_history(
C:\Users\AlexK\AppData\Local\Temp\ipykernel_30356\3622088872.py:11: RuntimeWarning: Usando historico PROVISORIO derivado de D+7. Isso contorna a excecao, mas nao comprova que quantity e venda diaria. Para resultados finais, passe history_df com as vendas diarias reais.
  daily, gaps_audit = prep.build_daily_history(


,category,target_date,min_quantity,max_quantity,num_horizons,difference
0,N02BE,2017-01-14,34.625,161.000,2,126.375
1,N02BE,2017-01-06,38.400,161.000,2,122.600
2,N02BE,2016-01-16,0.000,88.300,2,88.300
3,N02BE,2017-01-15,25.375,108.700,2,83.325
4,N02BE,2017-01-07,32.000,108.700,2,76.700
5,N02BE,2017-10-16,0.000,72.100,2,72.100
6,N02BE,2019-02-02,27.800,97.800,2,70.000
7,N02BE,2019-01-16,0.000,64.800,2,64.800
8,N02BE,2016-01-03,0.000,63.500,2,63.500
9,N02BE,2016-12-26,0.000,62.062,2,62.062


Histórico diário: (2051, 4)


,category,days,missing_days_filled,first_day,last_day,method,source,target_conflicts_total
0,N02BE,2051,0,2014-03-13,2019-10-23,ffill,proxy_D7_nas_datas_alvo,2013


Modo exploratório; ENFORCE_TARGET_ALIGNMENT = False


In [20]:
serie = daily[daily['category'].eq('N02BE')].sort_values('date')
expected = pd.date_range(serie['date'].min(), serie['date'].max(), freq='D')
missing_dates = expected.difference(pd.DatetimeIndex(serie['date']))

print('Lacunas restantes N02BE:', len(missing_dates))
print('Dias preenchidos N02BE:', int(serie['was_imputed'].sum()))
print('Quantidades ainda nulas N02BE:', int(serie['quantity'].isna().sum()))
display(serie[serie['was_imputed'].eq(1)].head(30))


Lacunas restantes N02BE: 0
Dias preenchidos N02BE: 0
Quantidades ainda nulas N02BE: 0


,date,category,quantity,was_imputed


In [21]:
# O df supervisionado NÃO é atualizado com a série diária.
# Para preservar D+7/D+15, use a chave completa incluindo horizon_days.
# A amostra de 2014-03-18 pode ter duas quantidades diferentes, e isso é esperado.
display(df.loc[(df['date'] == '2014-03-18') & (df['category'] == 'N02BE'),
               ['date', 'category', 'horizon_days', 'split', 'quantity']]
          .sort_values('horizon_days'))


,date,category,horizon_days,split,quantity
12,2014-03-18,N02BE,7,train,20.1
2047,2014-03-18,N02BE,15,train,20.1


In [22]:
# Checagem operacional: o gerador recebe histórico diário regular,
# mas divergências de targets são apenas AVISADAS neste modo exploratório.
df_validated = stacking._ensure_frame(df)
daily_validated = stacking._daily_history(df_validated)
print('Supervisionado:', df_validated.shape, 'Histórico:', daily_validated.shape)
display(daily_validated.head())


Supervisionado: (4078, 46) Histórico: (2051, 3)


C:\Users\AlexK\AppData\Local\Temp\ipykernel_30356\2343044375.py:4: RuntimeWarning: 2013 alvos supervisionados nao coincidem com o valor diario em date+h. Examine a definicao de quantity. Desabilitar ENFORCE_TARGET_ALIGNMENT apenas permite testes exploratorios, NAO valida as metricas OOF/holdout.
  daily_validated = stacking._daily_history(df_validated)


,date,category,quantity
0,2014-03-13,N02BE,44.1
1,2014-03-14,N02BE,25.0
2,2014-03-15,N02BE,27.6
3,2014-03-16,N02BE,11.3
4,2014-03-17,N02BE,47.0


## 2. Definir a frequência de backtesting

`ORIGIN_STEP=1` = previsões em todas as datas (mais lento); `7` = amostrar datas semanalmente para um primeiro teste. A mesma seleção de origens é aplicada aos quatro modelos. Os parâmetros de LightGBM/XGBoost e SARIMAX/AutoTheta ficam no topo do script, sem GridSearch feito no conjunto futuro.

In [23]:
stacking.N_SPLITS = 3
stacking.ORIGIN_STEP = 7  # teste mais rapido; ALTERE PARA 1 na geracao final
stacking.MIN_HISTORY = 60

print('Folds:', stacking.N_SPLITS)
print('Intervalo entre origens amostradas:', stacking.ORIGIN_STEP)
print('Modelos:', stacking.MODELS)


Folds: 3
Intervalo entre origens amostradas: 7
Modelos: ['lightgbm', 'xgboost', 'sarimax', 'autotheta']


## 3. Gerar os oito arquivos

Observação: sem `statsforecast` ou `pyarrow`, instale as dependências no `.venv`. SARIMAX pode levar tempo conforme o número de categorias ATC e folds.

In [24]:
outputs, audit = stacking.generate(df, persist=True)
display(audit)


D:\PROJETOS\git_repo\PharmaDemand-ml\src\models\generate_stacking_predictions.py:402: RuntimeWarning: 2013 alvos supervisionados nao coincidem com o valor diario em date+h. Examine a definicao de quantity. Desabilitar ENFORCE_TARGET_ALIGNMENT apenas permite testes exploratorios, NAO valida as metricas OOF/holdout.
  daily = _daily_history(data)


OOF fold 1: 2015-06-14 a 2016-09-20, 134 linhas
  lightgbm: 134 previsoes
  xgboost: 134 previsoes
  sarimax: 134 previsoes
  autotheta: 134 previsoes
OOF fold 2: 2016-09-21 a 2017-12-29, 134 linhas
  lightgbm: 134 previsoes
  xgboost: 134 previsoes
  sarimax: 134 previsoes
  autotheta: 134 previsoes
OOF fold 3: 2017-12-30 a 2019-04-08, 134 linhas
  lightgbm: 134 previsoes
  xgboost: 134 previsoes
  sarimax: 134 previsoes
  autotheta: 134 previsoes
Holdout: 2019-04-09 a 2019-10-08, 54 linhas
  lightgbm: 54 previsoes
  xgboost: 54 previsoes
  sarimax: 54 previsoes
  autotheta: 54 previsoes
SALVO: D:\PROJETOS\git_repo\PharmaDemand-ml\data\stacking\oof_lightgbm.parquet (402, 5)
SALVO: D:\PROJETOS\git_repo\PharmaDemand-ml\data\stacking\oof_xgboost.parquet (402, 5)
SALVO: D:\PROJETOS\git_repo\PharmaDemand-ml\data\stacking\oof_sarimax.parquet (402, 5)
SALVO: D:\PROJETOS\git_repo\PharmaDemand-ml\data\stacking\oof_autotheta.parquet (402, 5)
SALVO: D:\PROJETOS\git_repo\PharmaDemand-ml\data\stac

,kind,fold,horizon_days,last_available_target_date,validation_start,validation_end,fit_rows_tabular
0,oof,1,7,2015-06-13,2015-06-14,2016-09-20,458
1,oof,1,15,2015-06-13,2015-06-14,2016-09-20,442
2,oof,2,7,2016-09-20,2016-09-21,2017-12-29,923
3,oof,2,15,2016-09-20,2016-09-21,2017-12-29,907
4,oof,3,7,2017-12-29,2017-12-30,2019-04-08,1388
5,oof,3,15,2017-12-29,2017-12-30,2019-04-08,1372
6,holdout,0,7,2019-04-08,2019-04-09,2019-10-08,1853
7,holdout,0,15,2019-04-08,2019-04-09,2019-10-08,1837


## 4. Auditar alinhamento dos quatro modelos

In [16]:
for kind in ('oof', 'holdout'):
    a = outputs[kind]['lightgbm']
    print('\n', kind.upper(), '| linhas:', len(a), '| datas:', a.date.min(), 'ate', a.date.max())
    print('Categorias:', a.category.nunique(), '| Horizontes:', a.horizon_days.value_counts().to_dict())
    for name in stacking.MODELS:
        one = outputs[kind][name]
        print(f'  {kind}_{name}:', len(one), 'linhas; NaNs:', one.prediction.isna().sum())

display(outputs['oof']['lightgbm'].head(10))



 OOF | linhas: 396 | datas: 2015-06-23 00:00:00 ate 2019-04-02 00:00:00
Categorias: 1 | Horizontes: {7: 198, 15: 198}
  oof_lightgbm: 396 linhas; NaNs: 0
  oof_xgboost: 396 linhas; NaNs: 0
  oof_sarimax: 396 linhas; NaNs: 0
  oof_autotheta: 396 linhas; NaNs: 0

 HOLDOUT | linhas: 52 | datas: 2019-04-16 00:00:00 ate 2019-10-08 00:00:00
Categorias: 1 | Horizontes: {7: 26, 15: 26}
  holdout_lightgbm: 52 linhas; NaNs: 0
  holdout_xgboost: 52 linhas; NaNs: 0
  holdout_sarimax: 52 linhas; NaNs: 0
  holdout_autotheta: 52 linhas; NaNs: 0


,date,category,horizon_days,quantity,prediction
0,2015-06-23,N02BE,7,19.00,16.467727
1,2015-06-23,N02BE,15,19.00,22.040823
2,2015-06-30,N02BE,7,35.00,20.490522
3,2015-06-30,N02BE,15,35.00,21.535880
4,2015-07-07,N02BE,7,19.60,13.799330
5,2015-07-07,N02BE,15,19.60,14.145834
6,2015-07-14,N02BE,7,14.00,13.312338
7,2015-07-14,N02BE,15,14.00,15.476250
8,2015-07-21,N02BE,7,14.25,26.259630
9,2015-07-21,N02BE,15,14.25,23.181381


## 5. Antes de treinar e divulgar o Ridge

O objetivo da versão corrigida é **desbloquear o diagnóstico** e a geração experimental de oito Parquets. Se `target_conflicts()` encontra divergências, os resultados **não podem ser considerados validação confiável de previsões D+7/D+15** até identificar a verdadeira semântica da coluna `quantity` e alinhar as previsões dos quatro modelos com o mesmo alvo.

Para resultados finais: forneça o histórico de vendas `salesdaily` original, ajuste o ETL dos alvos para os horizontes corretos, confirme `ENFORCE_TARGET_ALIGNMENT=True` sem erros, depois execute `ORIGIN_STEP=1`.
